# Modern AABB quatrain generation (L40 + vLLM)

One notebook for both the test and the real run. Set `MODE` in the settings cell:

- **`"pilot"`**: about 300 prompts, a few minutes. Shows pass rates, failure reasons, good and bad samples, and a time estimate for the full run. Output goes to `pilot_v2_out/`.
- **`"full"`**: generates in chunks until `TARGET_USABLE` quatrains are saved to `gen_out/quatrains.jsonl`. Results are saved after every chunk, so if it crashes you can rerun the notebook and it resumes where it stopped.

Changes from the first pilot:
1. **Free rhymes only.** The fixed-end-word mode produced nonsense, so the model picks its own rhymes, and the rhyme pairs are read off each finished quatrain as the conditioning label.
2. **A prompt built around couplets.** Two worked examples (as chat turns) and a `RHYMES:` plan line that commits the model to AABB before it writes. The first pilot wrote ABAB most of the time.
3. **Several candidates per prompt** (`N_CANDIDATES`), keeping the best one that passes.
4. **Curly apostrophes normalised** before checking (`I’ll` was being split into `I` + `ll`).
5. **End words must be common words** (word frequency check), since CMUdict also contains names and odd spellings.
6. **Dataset-level filters:** exact and near-duplicate removal (ROUGE-L), a cap on each rhyme pair, and no reusing lines.
7. **A larger topic list** (about 600), generated once by the model and cached in `topics.json`.

**Workflow:** run in `"pilot"` mode → read the samples → adjust the prompt or settings if needed → set `MODE = "full"` → *Restart & Run All*.

## 1. Install

In [1]:
%pip install -q vllm pronouncing wordfreq pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Settings

**`MODE` is the switch.** Everything else can stay at its default for a first run.

In [2]:
import os, re, json, time, random
from collections import Counter, defaultdict

# ======================= THE SWITCH =======================
MODE = "pilot"              # "pilot" = small test with samples | "full" = real run, chunked and resumable
# ==========================================================
assert MODE in ("pilot", "full")

MODEL_ID = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"
# MODEL_ID = "google/gemma-3-27b-it"                  # needs a Hugging Face login and licence acceptance
# MODEL_ID = "Qwen/Qwen3-30B-A3B-Thinking-2507-FP8"   # reasons before writing: slower, follows rules better; set MAX_TOKENS ~1500

# --- run size ---
PILOT_PROMPTS = 300         # pilot: number of prompts
TARGET_USABLE = 20_000      # full: stop once this many quatrains are saved (try 50_000 later)
CHUNK_PROMPTS = 2_000       # full: prompts per chunk; results are saved after each chunk
MAX_CHUNKS    = 200         # full: safety stop

# --- sampling ---
N_CANDIDATES = 4            # samples per prompt; the best passing one is kept
TEMPERATURE  = 0.9
TOP_P        = 0.95
MAX_TOKENS   = 160          # RHYMES line + 4 lines is about 70-90 tokens
SEED         = 42

# --- quality filters ---
SYL_MIN, SYL_MAX = 6, 12    # allowed syllables per line
PAIR_SYL_DIFF    = 3        # max syllable difference within a couplet
MAX_OOV_RATE     = 0.15     # max share of words not in CMUdict
MIN_END_ZIPF     = 2.5      # end words must be at least this common (wordfreq Zipf scale); blocks names and made-up words
MAX_PAIR_USES    = 50       # max times one rhyme pair (e.g. name/game) may appear in the dataset
NEAR_DUP_ROUGE   = 0.7      # ROUGE-L F1 above this = near-duplicate

OUT_DIR     = "gen_out" if MODE == "full" else "pilot_v2_out"
TOPICS_FILE = "topics.json" # shared by pilot and full runs
os.makedirs(OUT_DIR, exist_ok=True)
print("MODE:", MODE, "| output:", OUT_DIR)

MODE: pilot | output: pilot_v2_out


## 3. Phonetic helpers

In [3]:
import pronouncing
from wordfreq import zipf_frequency

WORD_RE   = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
QUOTE_MAP = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'})

def normalize(text):
    return text.translate(QUOTE_MAP)        # curly quotes -> straight, so "I’ll" stays one word

def words_of(line):
    return [w.lower() for w in WORD_RE.findall(normalize(line))]

def last_word(line):
    w = words_of(line)
    return w[-1] if w else ""

def all_phones(word):
    return pronouncing.phones_for_word(word.lower())

def rhyme_parts(word):
    return {pronouncing.rhyming_part(p) for p in all_phones(word)}

def same_sound(a, b):
    # True if a and b share a rhyme sound (identical words count)
    ra, rb = rhyme_parts(a), rhyme_parts(b)
    return bool(ra and rb and ra & rb)

def rhymes(a, b):
    # a real rhyme: same sound, different words, and not one word hiding inside the other (play / replay)
    a, b = a.lower(), b.lower()
    if not a or not b or a == b or a.endswith(b) or b.endswith(a):
        return False
    return same_sound(a, b)

def word_syllables(word):
    ph = all_phones(word)
    if ph:
        return pronouncing.syllable_count(ph[0])
    groups = re.findall(r"[aeiouy]+", word.lower())       # fallback for unknown words
    n = len(groups)
    if word.lower().endswith("e") and n > 1 and not word.lower().endswith("le"):
        n -= 1
    return max(n, 1)

def line_syllables(line):
    return sum(word_syllables(w) for w in words_of(line))

def oov_rate(lines):
    ws = [w for l in lines for w in words_of(l)]
    return sum(1 for w in ws if not all_phones(w)) / len(ws) if ws else 1.0

assert rhymes("late", "wait") and rhymes("phone", "alone") and rhymes("sun", "one")
assert not rhymes("late", "phone") and not rhymes("play", "replay")
assert words_of("I’ll go") == ["i'll", "go"]
print("helpers ok")

helpers ok


## 4. Load the model

In [4]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    enable_prefix_caching=True,   # the system prompt and examples are shared, so they are cached
    seed=SEED,
)
params = SamplingParams(n=N_CANDIDATES, temperature=TEMPERATURE, top_p=TOP_P,
                        max_tokens=MAX_TOKENS, repetition_penalty=1.05)

/common/home/projectgrps/CS480/CS480G3/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(MainProcess pid=3246751) INFO 10-20 09:51:39 [api_utils.py:286] non-default args: {'seed': 42, 'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507-FP8'}
(MainProcess pid=3246751) INFO 10-20 09:51:41 [model.py:733] Resolved architecture: Qwen3MoeForCausalLM
(MainProcess pid=3246751) INFO 10-20 09:51:41 [model.py:2152] Using max model len 4096
(MainProcess pid=3246751) INFO 10-20 09:51:43 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=3246751) INFO 10-20 09:51:43 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=3246751) INFO 10-20 09:51:43 [compilation.py:317] Enabled custom fusions: norm_quant, act_quant


Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 1201.08it/s]


(EngineCore pid=3246898) INFO 10-20 09:51:44 [core.py:133] Initializing a V1 LLM engine (v0.32.0) with config: model='Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', speculative_config=None, tokenizer='Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=fp8, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:04<00:14,  4.77s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:09<00:09,  4.78s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:14<00:04,  4.77s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:15<00:00,  3.17s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:15<00:00,  3.76s/it]



(EngineCore pid=3246898) INFO 10-20 09:52:03 [default_loader.py:501] Loading weights took 15.06 seconds
(EngineCore pid=3246898) WARNING 10-20 09:52:04 [marlin_utils_fp8.py:108] Your GPU does not have native support for FP8 computation but FP8 quantization is being used. Weight-only FP8 compression will be used leveraging the Marlin kernel. This may degrade performance for compute-heavy workloads.
(EngineCore pid=3246898) INFO 10-20 09:52:04 [fp8.py:856] Using MoEPrepareAndFinalizeNoDPEPModular
(EngineCore pid=3246898) INFO 10-20 09:52:05 [model_runner.py:424] Model loading took 29.11 GiB memory and 19.600596 seconds
(EngineCore pid=3246898) INFO 10-20 09:52:05 [topk_topp_sampler.py:124] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=3246898) INFO 10-20 09:52:05 [torch_utils.py:295] Reducing Torch threads from 4 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=3246898) INFO 10-20 09:52:05 [utils.py:347] Using LBNHC KV cache la

Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00, 16.59it/s]


(EngineCore pid=3246898) INFO 10-20 09:52:45 [model_runner.py:1127] Graph capturing finished in 7 secs, took 0.59 GiB
(EngineCore pid=3246898) INFO 10-20 09:52:46 [gpu_worker.py:759] Available KV cache memory: 8.95 GiB
(EngineCore pid=3246898) INFO 10-20 09:52:46 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9000 is equivalent to --gpu-memory-utilization=0.8857 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9143. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=3246898) INFO 10-20 09:52:46 [kv_cache_utils.py:2327] GPU KV cache size: 97,760 tokens, Maximum concurrency for 4,096 tokens per request: 23.87x
(EngineCore pid=3246898) INFO 10-20 09:52:46 [kernel_warmup.py:176] JIT kernel warmup starting.
(EngineCore pid=3246898) INFO 10-20 09:52:46 [kernel_warmup.py:189] JIT kernel warmup finished in 0.00s.

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 17.54it/s]


(EngineCore pid=3246898) INFO 10-20 09:52:55 [model_runner.py:1127] Graph capturing finished in 7 secs, took 0.33 GiB
(EngineCore pid=3246898) INFO 10-20 09:52:55 [gpu_worker.py:971] CUDA graph pool memory: 0.33 GiB (actual), 0.64 GiB (estimated), difference: 0.31 GiB (92.9%).
(EngineCore pid=3246898) INFO 10-20 09:52:55 [gpu_worker.py:1034] Free memory on device (43.97/44.39 GiB) on startup. Desired GPU memory utilization is (0.9, 39.95 GiB). Actual usage is 30.0 GiB for consumed memory (weights + non-torch), 1.0 GiB for peak activation, and 0.33 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=9099065037` (8.47 GiB) to fit into requested memory, or `--kv-cache-memory=13410494976` (12.49 GiB) to fully utilize gpu memory. Current kv cache memory in use is 8.95 GiB.
(EngineCore pid=3246898) INFO 10-20 09:52:56 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=3246898) I

## 5. Topics

About 600 everyday topics: the model writes 30 per category once, and they are cached in `topics.json`. Delete that file to regenerate, or edit it by hand to remove weak topics.

In [5]:
BASE_TOPICS = [
    "missing the morning bus", "a group project where nobody helps", "the last slice of pizza",
    "a phone at one percent battery", "Monday mornings", "a cat that ignores you", "rainy weekends",
    "a gym membership you never use", "waiting for exam results", "a noisy neighbour",
    "online shopping regrets", "a houseplant that keeps dying", "the first day at a new job",
    "a friend who is always late", "burnt toast", "a long queue at the coffee shop",
    "forgetting someone's name", "the summer heat", "a road trip with bad directions",
    "a video call with a frozen screen", "a messy bedroom", "learning to cook", "a broken umbrella",
    "staying up too late", "a lost set of keys", "a birthday nobody remembered", "a cheap haircut",
    "sending a text to the wrong person", "the alarm clock", "a crowded train", "homework due tomorrow",
    "a dog that steals socks", "a failed diet", "an awkward first date", "the office printer",
    "a slow wifi connection", "grocery shopping while hungry", "a song stuck in your head",
    "moving to a new city", "a rainy football match", "a talkative taxi driver", "a group chat",
    "trying to wake up early", "a surprise party", "a spilled drink", "the weekend ending too soon",
    "a long flight", "a boring meeting", "a forgotten password", "the smell of fresh bread",
]
TOPIC_CATEGORIES = [
    "work and the office", "school and university", "cooking and food", "phones, apps and the internet",
    "pets", "weather and seasons", "public transport and driving", "family life", "friendship",
    "dating and relationships", "sport and the gym", "health and sleep", "money and bills",
    "home, chores and neighbours", "shopping", "holidays and travel", "social media",
    "birthdays and celebrations", "music and films", "hobbies", "city life", "nature and the outdoors",
    "childhood memories", "getting older", "small daily annoyances",
]

def generate_topics():
    msgs = [[{"role": "user", "content":
              f"List 30 specific everyday situations about {cat} that a short, light modern poem could be about. "
              "Each one is 3 to 8 words and concrete, like 'a cat that ignores you' or 'missing the morning bus'. "
              "Return only a JSON array of strings."}] for cat in TOPIC_CATEGORIES]
    outs = llm.chat(msgs, SamplingParams(temperature=0.8, top_p=0.95, max_tokens=900), use_tqdm=False)
    found = []
    for o in outs:
        text = o.outputs[0].text
        m = re.search(r"\[.*\]", text, re.S)
        try:
            found += [t for t in json.loads(m.group(0)) if isinstance(t, str)]
        except Exception:                     # fall back to one topic per line
            found += [re.sub(r'^[\s\-*\d.)"]+|[",]+$', "", l) for l in text.splitlines()]
    return found

if os.path.exists(TOPICS_FILE):
    TOPICS = json.load(open(TOPICS_FILE, encoding="utf-8"))
else:
    seen, TOPICS = set(), []
    for t in BASE_TOPICS + generate_topics():
        t = normalize(t).strip().rstrip(".")
        if 2 <= len(t.split()) <= 10 and t.lower() not in seen:
            seen.add(t.lower()); TOPICS.append(t)
    json.dump(TOPICS, open(TOPICS_FILE, "w", encoding="utf-8"), indent=1)

TONES = ["funny", "sarcastic", "wistful", "hopeful", "deadpan", "dramatic", "playful", "grumpy",
         "tender", "nostalgic", "anxious", "cheerful", "bittersweet", "proud"]
VOICES = ["first person (I)", "second person (you)", "a narrator describing someone"]
print(f"{len(TOPICS)} topics, e.g.", random.Random(0).sample(TOPICS, 6))

797 topics, e.g. ['itchy eyes from screens', 'glue stick dried out', 'receipts scattered on the table', 'a group chat', 'kitchen light left on', 'restaurant with no menu in English']


## 6. Prompt

Two of the four worked examples are given as earlier chat turns, chosen at random for each prompt. The model first writes a `RHYMES:` line with its end words, which commits it to two couplets. The parser removes that line.

In [6]:
SYSTEM = (
    "You write short modern poems in rhyming couplets. Reply with a RHYMES line followed by exactly "
    "4 lines of poetry. No title, no numbering, no blank lines, no explanation."
)

EXAMPLES = [
    dict(topic="a phone at one percent battery", tone="dramatic", voice="first person (I)",
         rhymes="red / bed ; in / win",
         lines=["My battery icon has just turned red,",
                "I search for a charger under the bed.",
                "I find one at last and I plug it right in,",
                "the cable is broken, I just can't win."]),
    dict(topic="the office printer", tone="grumpy", voice="a narrator describing someone",
         rhymes="nine / line ; tray / day",
         lines=["He needs ten copies by a quarter past nine,",
                "the printer just blinks at the end of the line.",
                "He opens each drawer and he empties each tray,",
                "then writes them by hand for the rest of the day."]),
    dict(topic="rainy weekends", tone="wistful", voice="second person (you)",
         rhymes="sun / one ; street / sweet",
         lines=["You had big plans for the park and the sun,",
                "the rain cancelled each of them one by one.",
                "So you make some tea and you watch the street,",
                "and find that a slow day is almost as sweet."]),
    dict(topic="a group chat", tone="sarcastic", voice="first person (I)",
         rhymes="night / right ; all / small",
         lines=["The group chat pings a hundred times at night,",
                "four people arguing about who's right.",
                "I scroll back up to see what started it all:",
                "a photo of a sandwich, slightly small."]),
]

def user_prompt(topic, tone, voice):
    return (
        "Write one quatrain made of two rhyming couplets (AABB).\n"
        "- Line 1 rhymes with line 2. Line 3 rhymes with line 4.\n"
        "- The second couplet uses a different rhyme sound. Line 1 must not rhyme with line 3.\n"
        "- 7 to 11 syllables per line.\n"
        "- Modern everyday English. No archaic words, no cliches, simple punctuation.\n"
        "- Avoid overused rhymes like night/light, heart/apart, day/way.\n"
        f"Topic: {topic}\nTone: {tone}\nVoice: {voice}\n"
        "First write the RHYMES line with the end words of each couplet, then the 4 lines."
    )

def build_messages(topic, tone, voice, rng):
    msgs = [{"role": "system", "content": SYSTEM}]
    for ex in rng.sample(EXAMPLES, 2):
        msgs.append({"role": "user", "content": user_prompt(ex["topic"], ex["tone"], ex["voice"])})
        msgs.append({"role": "assistant", "content": "RHYMES: " + ex["rhymes"] + "\n" + "\n".join(ex["lines"])})
    msgs.append({"role": "user", "content": user_prompt(topic, tone, voice)})
    return msgs

def make_jobs(n, rng, start_id):
    jobs = []
    for i in range(n):
        topic, tone, voice = rng.choice(TOPICS), rng.choice(TONES), rng.choice(VOICES)
        jobs.append({"id": start_id + i, "topic": topic, "tone": tone, "voice": voice,
                     "messages": build_messages(topic, tone, voice, rng)})
    return jobs

print(user_prompt("a lost set of keys", "funny", VOICES[0]))

Write one quatrain made of two rhyming couplets (AABB).
- Line 1 rhymes with line 2. Line 3 rhymes with line 4.
- The second couplet uses a different rhyme sound. Line 1 must not rhyme with line 3.
- 7 to 11 syllables per line.
- Modern everyday English. No archaic words, no cliches, simple punctuation.
- Avoid overused rhymes like night/light, heart/apart, day/way.
Topic: a lost set of keys
Tone: funny
Voice: first person (I)
First write the RHYMES line with the end words of each couplet, then the 4 lines.


## 7. Checks and dataset filters

**Per candidate:** 4 lines, AABB rhyme (ABAB is reported separately), different A and B sounds, syllable range, balanced couplets, unknown-word rate, rare or made-up end words.

**Against the dataset so far:** exact duplicate, rhyme-pair cap, reused lines, near-duplicate (ROUGE-L, compared only with quatrains sharing a rhyme pair, so it stays fast at 50k).

In [7]:
THINK_RE = re.compile(r"<think>.*?</think>", re.S)

def parse(raw):
    text = normalize(THINK_RE.sub("", raw))
    plan, lines = None, []
    for l in text.strip().splitlines():
        l = l.strip()
        if len(l) > 1 and l[0] == l[-1] == '"':
            l = l[1:-1].strip()
        if not l:
            continue
        if l.upper().startswith("RHYMES"):
            plan = l.split(":", 1)[-1].strip()
            continue
        l = re.sub(r"^(\d+[\.\):]|[-*•])\s*", "", l)       # drop numbering / bullets
        if l.lower().startswith(("here", "title", "sure")):
            continue
        lines.append(l)
    return plan, lines

def check(lines):
    r = {"lines": lines, "ends": None, "syllables": None, "oov": None, "reasons": []}
    if len(lines) != 4:
        r["reasons"].append(f"line_count={len(lines)}")
    else:
        ends = [last_word(l) for l in lines]
        r["ends"] = ends
        a_ok, b_ok = rhymes(ends[0], ends[1]), rhymes(ends[2], ends[3])
        if not a_ok and not b_ok and same_sound(ends[0], ends[2]) and same_sound(ends[1], ends[3]):
            r["reasons"].append("ABAB")
        else:
            if not a_ok: r["reasons"].append("A_no_rhyme")
            if not b_ok: r["reasons"].append("B_no_rhyme")
            if a_ok and b_ok and same_sound(ends[0], ends[2]): r["reasons"].append("same_sound_AB")
        syl = [line_syllables(l) for l in lines]
        r["syllables"] = syl
        if any(s < SYL_MIN or s > SYL_MAX for s in syl):
            r["reasons"].append("syllable_range")
        if abs(syl[0] - syl[1]) > PAIR_SYL_DIFF or abs(syl[2] - syl[3]) > PAIR_SYL_DIFF:
            r["reasons"].append("pair_unbalanced")
        if any(zipf_frequency(e, "en") < MIN_END_ZIPF for e in ends):
            r["reasons"].append("rare_end_word")
        r["oov"] = round(oov_rate(lines), 3)
        if r["oov"] > MAX_OOV_RATE:
            r["reasons"].append("oov")
    r["form_pass"] = r["pass"] = not r["reasons"]
    return r

def near_dup(a, ca, b, cb):
    # ROUGE-L F1 > threshold? Word overlap is an upper bound on the LCS, so most pairs skip the DP.
    f1 = lambda n: 0.0 if n == 0 else 2 * n / (len(a) + len(b))
    if f1(sum((ca & cb).values())) <= NEAR_DUP_ROUGE:
        return False
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return f1(prev[-1]) > NEAR_DUP_ROUGE

class Corpus:
    """Accepted quatrains plus the indexes used for duplicate and rhyme-pair checks."""
    def __init__(self):
        self.items, self.keys = [], set()
        self.pair_uses, self.line_seen = Counter(), Counter()
        self.by_pair = defaultdict(list)          # rhyme pair -> (tokens, Counter) of accepted quatrains

    @staticmethod
    def pairs(ends):
        return [tuple(sorted(ends[0:2])), tuple(sorted(ends[2:4]))]

    def reject_reason(self, lines, ends):
        if " / ".join(l.lower() for l in lines) in self.keys:
            return "duplicate"
        ps = self.pairs(ends)
        if any(self.pair_uses[p] >= MAX_PAIR_USES for p in ps):
            return "pair_cap"
        if sum(self.line_seen[l.lower()] > 0 for l in lines) >= 2:
            return "reused_lines"
        toks = [w for l in lines for w in words_of(l)]
        c = Counter(toks)
        if any(near_dup(toks, c, t, ct) for p in ps for t, ct in self.by_pair[p]):
            return "near_duplicate"
        return None

    def add(self, rec, store=True):
        lines, ends = rec["lines"], [last_word(l) for l in rec["lines"]]
        self.keys.add(" / ".join(l.lower() for l in lines))
        toks = [w for l in lines for w in words_of(l)]
        for p in self.pairs(ends):
            self.pair_uses[p] += 1
            self.by_pair[p].append((toks, Counter(toks)))
        for l in lines:
            self.line_seen[l.lower()] += 1
        if store:
            self.items.append(rec)

def pick(cands, corpus):
    ok = []
    for c in cands:
        if not c["form_pass"]:
            continue
        why = corpus.reject_reason(c["lines"], c["ends"])
        if why:
            c["pass"] = False; c["reasons"].append(why)
        else:
            ok.append(c)
    if not ok:
        return None
    # prefer the candidate whose rhyme pairs are least used so far, which spreads the rhyme vocabulary
    return min(ok, key=lambda c: max(corpus.pair_uses[p] for p in corpus.pairs(c["ends"])))

def run_chunk(jobs, corpus):
    t0 = time.time()
    outs = llm.chat([j["messages"] for j in jobs], params, use_tqdm=True)
    secs, toks, kept = time.time() - t0, 0, []
    for j, o in zip(jobs, outs):
        j["cands"] = []
        for c in o.outputs:
            toks += len(c.token_ids)
            plan, lines = parse(c.text)
            r = check(lines)
            r.update(raw=c.text, plan=plan)
            j["cands"].append(r)
        best = pick(j["cands"], corpus)
        if best:
            e = best["ends"]
            rec = {"id": j["id"], "topic": j["topic"], "tone": j["tone"], "voice": j["voice"],
                   "lines": best["lines"], "rhymes": [e[0:2], e[2:4]],
                   "syllables": best["syllables"], "model": MODEL_ID}
            corpus.add(rec)
            kept.append(rec)
    return kept, secs, toks

def new_corpus():
    corpus = Corpus()
    for ex in EXAMPLES:                    # so copies of the worked examples get rejected
        corpus.add({"lines": ex["lines"]}, store=False)
    return corpus
print("checks ok")

checks ok


## 8. Run

Pilot mode runs one batch. Full mode runs chunk after chunk and appends to `gen_out/quatrains.jsonl`. If it stops for any reason, rerun the whole notebook: it reloads what is saved and continues from the next chunk.

In [8]:
if MODE == "pilot":
    corpus = new_corpus()
    jobs = make_jobs(PILOT_PROMPTS, random.Random(SEED), 0)
    kept, secs, toks = run_chunk(jobs, corpus)
    print(f"\n{len(jobs)} prompts x {N_CANDIDATES} candidates in {secs:.1f}s | {toks:,} output tokens "
          f"({toks/secs:,.0f} tok/s) | kept {len(kept)}")
    run_stats = {"prompts": len(jobs), "kept": len(kept), "seconds": secs, "tokens": toks}

else:
    KEPT_FILE, LOG_FILE = f"{OUT_DIR}/quatrains.jsonl", f"{OUT_DIR}/progress.jsonl"
    corpus = new_corpus()
    done = sum(1 for _ in open(LOG_FILE)) if os.path.exists(LOG_FILE) else 0
    if os.path.exists(KEPT_FILE):
        saved = [json.loads(l) for l in open(KEPT_FILE, encoding="utf-8")]
        good = [r for r in saved if r["chunk"] < done]     # drop a half-saved chunk from a crash
        if len(good) != len(saved):
            with open(KEPT_FILE, "w", encoding="utf-8") as f:
                f.writelines(json.dumps(r) + "\n" for r in good)
        for r in good:
            corpus.add(r)
        print(f"Resuming after chunk {done}: {len(corpus.items):,} quatrains already saved")

    run_stats = {"prompts": 0, "kept": 0, "seconds": 0.0, "tokens": 0}
    jobs, chunk = [], done
    while len(corpus.items) < TARGET_USABLE and chunk < MAX_CHUNKS:
        before = len(corpus.items)
        jobs = make_jobs(CHUNK_PROMPTS, random.Random(SEED * 100_003 + chunk), chunk * CHUNK_PROMPTS)
        kept, secs, toks = run_chunk(jobs, corpus)
        kept = kept[:TARGET_USABLE - before]              # don't overshoot the target
        del corpus.items[before + len(kept):]
        for r in kept:
            r["chunk"] = chunk
        with open(KEPT_FILE, "a", encoding="utf-8") as f:
            f.writelines(json.dumps(r) + "\n" for r in kept)
        reasons = Counter(x for j in jobs for c in j["cands"] for x in c["reasons"])
        with open(LOG_FILE, "a") as f:
            f.write(json.dumps({"chunk": chunk, "prompts": len(jobs), "kept": len(kept), "seconds": round(secs, 1),
                                "tokens": toks, "total": len(corpus.items), "reasons": dict(reasons)}) + "\n")
        for k, v in (("prompts", len(jobs)), ("kept", len(kept)), ("seconds", secs), ("tokens", toks)):
            run_stats[k] += v
        print(f"chunk {chunk}: kept {len(kept)}/{len(jobs)} ({len(kept)/len(jobs):.0%}) in {secs:.0f}s | "
              f"total {len(corpus.items):,}/{TARGET_USABLE:,} | top reasons {reasons.most_common(3)}")
        chunk += 1
        if len(kept) < 0.05 * len(jobs) and len(corpus.items) < TARGET_USABLE:
            print("Yield fell below 5% of prompts, stopping. Check the reasons above: usually pair_cap "
                  "(raise MAX_PAIR_USES or add topics) or near_duplicate (raise TEMPERATURE).")
            break
    print(f"\nDone: {len(corpus.items):,} quatrains in {KEPT_FILE}")

Processed prompts: 100%|██████████| 1200/1200 [00:18<00:00, 64.19it/s, est. speed input: 40114.94 toks/s, output: 3886.39 toks/s] 



300 prompts x 4 candidates in 19.0s | 72,656 output tokens (3,821 tok/s) | kept 132


## 9. Stats

In full mode, the candidate-level numbers are for the last chunk and the dataset-level numbers cover everything saved.

In [9]:
import pandas as pd

cands = [c for j in jobs for c in j["cands"]]
if cands:
    form_ok = sum(c["form_pass"] for c in cands)
    kept_prompts = sum(any(c["pass"] for c in j["cands"]) for j in jobs)
    print(f"Candidates passing form checks: {form_ok}/{len(cands)} ({form_ok/len(cands):.0%})")
    print(f"Prompts with a kept quatrain:   {kept_prompts}/{len(jobs)} ({kept_prompts/len(jobs):.0%})\n")
    print("Failure reasons per candidate (one can fail for several):")
    for r, n in Counter(x for c in cands for x in c["reasons"]).most_common():
        print(f"  {r:16s} {n:6d}  ({n/len(cands):.0%})")

items = corpus.items
syl = [s for r in items for s in r["syllables"]]
if syl:
    print(f"\nSyllables per line (kept): mean {sum(syl)/len(syl):.1f}, min {min(syl)}, max {max(syl)}")
    print("  distribution:", dict(sorted(Counter(syl).items())))

pairs = Counter(tuple(p) for r in items for p in r["rhymes"])
print(f"\nDataset: {len(items):,} quatrains | {len(pairs):,} distinct rhyme pairs")
print("Most used pairs:", pairs.most_common(10))
sounds = Counter(min(rhyme_parts(p[0]) or {"?"}) for r in items for p in r["rhymes"])
print(f"Distinct rhyme sounds: {len(sounds)}; most common:", sounds.most_common(6))

ws = [w for r in items for l in r["lines"] for w in words_of(l)]
if ws:
    print(f"Vocabulary: {len(set(ws)):,} distinct words in {len(ws):,} tokens (type/token {len(set(ws))/len(ws):.2f})")
print("Kept by tone:", dict(Counter(r["tone"] for r in items).most_common()))

Candidates passing form checks: 172/1200 (14%)
Prompts with a kept quatrain:   132/300 (44%)

Failure reasons per candidate (one can fail for several):
  B_no_rhyme          775  (65%)
  A_no_rhyme          761  (63%)
  syllable_range      203  (17%)
  pair_unbalanced      49  (4%)
  rare_end_word        19  (2%)
  ABAB                  4  (0%)
  same_sound_AB         3  (0%)
  line_count=3          1  (0%)

Syllables per line (kept): mean 10.0, min 6, max 12
  distribution: {6: 1, 7: 4, 8: 45, 9: 106, 10: 225, 11: 106, 12: 41}

Dataset: 132 quatrains | 232 distinct rhyme pairs
Most used pairs: [(('down', 'town'), 4), (('hold', 'bold'), 3), (('door', 'floor'), 3), (('gone', 'on'), 3), (('mess', 'yes'), 3), (('wrong', 'song'), 3), (('sound', 'found'), 2), (('glow', 'show'), 2), (('down', 'crown'), 2), (('near', 'clear'), 2)]
Distinct rhyme sounds: 87; most common: [('AO1 R', 13), ('OW1', 10), ('IH1 R', 10), ('AW1 N D', 8), ('OW1 L D', 8), ('EH1 S', 8)]
Vocabulary: 1,229 distinct words i

## 10. Samples

In [10]:
def show(rec_or_cand, head=""):
    print(head)
    for l, s in zip(rec_or_cand["lines"], rec_or_cand.get("syllables") or [None] * 8):
        print(f"   {l}" + (f"   ({s})" if s else ""))
    if rec_or_cand.get("reasons"):
        print("   FAIL:", ", ".join(rec_or_cand["reasons"]))
    print()

rng = random.Random()
batch_ids = {j["id"] for j in jobs}
kept_now = [r for r in corpus.items if r["id"] in batch_ids]
print("=========== KEPT (random 12) ===========\n")
for r in rng.sample(kept_now, min(12, len(kept_now))):
    show(r, f"[{r['id']}] {r['topic']} | {r['tone']} | {r['voice']} | rhymes {r['rhymes']}")

failed = [(j, c) for j in jobs for c in j["cands"] if not c["form_pass"]]
print("=========== FAILED FORM CHECKS (random 8) ===========\n")
for j, c in rng.sample(failed, min(8, len(failed))):
    show(c, f"[{j['id']}] {j['topic']} | {j['tone']} | plan: {c['plan']}")

=========== KEPT (random 12) ===========

[48] bus driver sleeping | deadpan | a narrator describing someone | rhymes [['wheel', 'meal'], ['still', 'drill']]
   The bus driver dozes at the wheel,   (9)
   no one complains, it's just a meal.   (8)
   The route is long, the stop is still,   (8)
   they'll wake when the next horn's on the drill.   (9)

[116] a self-checkout glitch | nostalgic | first person (I) | rhymes [['phone', 'tone'], ['track', 'back']]
   The screen froze when I tried to pay my phone,   (10)
   no scan, no receipt, just a dumb "error" tone.   (11)
   I remembered when machines still worked in track,   (11)
   and we'd hand cash to someone instead of a back.   (12)

[74] clouds gathering on a summer afternoon | tender | second person (you) | rhymes [['blue', 'through'], ['slow', 'go']]
   You watch the sky turn deep in shades of blue,   (10)
   as wind begins to hum through trees and through.   (10)
   The air grows still, and every step feels slow,   (11)
   like ti

## 11. Estimate and save

In [11]:
secs = max(run_stats["seconds"], 1e-6)
yield_per_prompt = run_stats["kept"] / max(run_stats["prompts"], 1)
prompts_per_sec = run_stats["prompts"] / secs
print(f"Yield: {yield_per_prompt:.0%} of prompts give a kept quatrain | {prompts_per_sec:.1f} prompts/s "
      f"| {run_stats['tokens']/secs:,.0f} tok/s")
if MODE == "pilot" and yield_per_prompt > 0:
    for target in (20_000, 50_000):
        n = target / yield_per_prompt
        print(f"  {target:,} quatrains -> about {n:,.0f} prompts -> roughly {n/prompts_per_sec/60:.0f} min")
    print("Larger chunks run faster per prompt, but yield drops a little as rhyme-pair caps fill, so treat this as a rough guide.")

def dump(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r) + "\n" for r in rows)

if MODE == "pilot":
    dump(f"{OUT_DIR}/pilot_kept.jsonl", corpus.items)
    dump(f"{OUT_DIR}/pilot_all.jsonl", [{k: v for k, v in j.items() if k != "messages"} for j in jobs])
meta = {"mode": MODE, "model": MODEL_ID, "n_candidates": N_CANDIDATES, "temperature": TEMPERATURE,
        "top_p": TOP_P, "saved": len(corpus.items), **run_stats}
json.dump(meta, open(f"{OUT_DIR}/meta.json", "w"), indent=2)
print("Saved to", OUT_DIR)

Yield: 44% of prompts give a kept quatrain | 15.8 prompts/s | 3,821 tok/s
  20,000 quatrains -> about 45,455 prompts -> roughly 48 min
  50,000 quatrains -> about 113,636 prompts -> roughly 120 min
Larger chunks run faster per prompt, but yield drops a little as rhyme-pair caps fill, so treat this as a rough guide.
Saved to pilot_v2_out


## What to check before switching to full

- **ABAB share** in the failure reasons. The first pilot was about 90% ABAB in free mode. If it is still high, the prompt needs more work (or try the Thinking model).
- **Read the 12 kept samples.** The filters only check form, so judge whether they read naturally and stay on topic.
- **Copies of the examples:** check that kept quatrains are not reworded versions of the four worked examples (red/bed, nine/line, sun/one, night/right).
- **Most used rhyme pairs.** If a few pairs dominate, the cap will start rejecting a lot in the full run.
- **Time estimate.** If it looks fine, set `MODE = "full"` in cell 2 and use *Restart & Run All*.

Output format (`gen_out/quatrains.jsonl`), one per line:
`{"id", "topic", "tone", "voice", "lines": [4 strings], "rhymes": [["late","wait"],["phone","alone"]], "syllables", "model", "chunk"}`.
`rhymes` is the conditioning label for the generator, read from the quatrain itself.